In [3]:
import os 
from dotenv import load_dotenv

load_dotenv()

print("Groq key set:",bool(os.environ["GROQ_API_KEY"]))

Groq key set: True


In [5]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
# response = model.invoke("Say hello in malayalam")
# print(response.content)
print(model.invoke("Reply with just the word: ready").content)

ready


Create a small sandbox folder for the Filesystem server

In [6]:
import pathlib

SANDBOX_DIR = pathlib.Path("mcp_sandbox").resolve()
SANDBOX_DIR.mkdir(exist_ok=True)

(SANDBOX_DIR / "notes.txt").write_text(
    "Project kickoff meeting is scheduled for next Monday at 10am.\n"
    "Attendees: Priya, Rahul, Aisha.\n"
)
print("Sandbox ready at:", SANDBOX_DIR)
print("Contents:", list(SANDBOX_DIR.iterdir()))

Sandbox ready at: /home/ronald/Desktop/python-pratice/ai-training/week-6/mcp_sandbox
Contents: [PosixPath('/home/ronald/Desktop/python-pratice/ai-training/week-6/mcp_sandbox/notes.txt')]


MCP Filesystem server to list available tools

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

filesystem_server_params = StdioServerParameters(
    command="npx",
    # -y auto-confirms the npm install prompt on first run
    args=["-y", "@modelcontextprotocol/server-filesystem", str(SANDBOX_DIR)],
)

async def list_filesystem_tools():
    async with stdio_client(filesystem_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print(f"Filesystem server exposes {len(tools)} tools:\n")
            for t in tools:
                print(f"  - {t.name}: {(t.description or '')[:80]}")

await list_filesystem_tools()

MCP Fetch server to list available tools

In [8]:
fetch_server_params = StdioServerParameters(
    command="python3",
    args=["-m", "mcp_server_fetch"],
)

async def list_fetch_tools():
    async with stdio_client(fetch_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print(f"Fetch server exposes {len(tools)} tool(s):\n")
            for t in tools:
                print(f"  - {t.name}: {(t.description or '')[:100]}")

await list_fetch_tools()

Fetch server exposes 1 tool(s):

  - fetch: Fetches a URL from the internet and optionally extracts its contents as markdown.

Although original


MCP Git server to list available tools

In [13]:
import sys
import subprocess
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

def get_current_repo_root() -> str:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    )
    return result.stdout.strip()

repo_path = get_current_repo_root()
print("Detected repo root:", repo_path)

git_server_params = StdioServerParameters(
    command=sys.executable,          # <-- was "python3"
    args=["-m", "mcp_server_git", "--repository", repo_path],
)

async def list_git_tools():
    async with stdio_client(git_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print(f"Git server exposes {len(tools)} tool(s):\n")
            for t in tools:
                print(f"  - {t.name}: {(t.description or '')[:100]}")

await list_git_tools()

Detected repo root: /home/ronald/Desktop/python-pratice/ai-training


  + Exception Group Traceback (most recent call last):
  |   File "/home/ronald/Desktop/python-pratice/ai-training/week-6/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3821, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/tmp/ipykernel_23706/468402719.py", line 31, in <module>
  |     await list_git_tools()
  |   File "/tmp/ipykernel_23706/468402719.py", line 23, in list_git_tools
  |     async with stdio_client(git_server_params) as (read, write):
  |   File "/usr/lib/python3.12/contextlib.py", line 231, in __aexit__
  |     await self.gen.athrow(value)
  |   File "/home/ronald/Desktop/python-pratice/ai-training/week-6/.venv/lib/python3.12/site-packages/mcp/client/stdio/__init__.py", line 182, in stdio_client
  |     async with (
  |   File "/home/ronald/Desktop/python-pratice/ai-training/week-6/.venv/lib/python3.12/site-packages/anyio/_backends/_asyncio.py", line 847, in __aexit__
  |     raise BaseExceptionGroup(


## ReAct agent with MCP tools

In [21]:
from langchain.agents import create_agent

async def run_filesystem_agent(question: str):
    async with stdio_client(filesystem_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)

            agent = create_react_agent(model, tools)
            result = await agent.ainvoke({"messages": question})
            print(result["messages"][-1].content)

await run_filesystem_agent(
    "List the files in the allowed directory, then read notes.txt and tell me "
    "who the attendees of the meeting are.",
    # "Update attendees Rahul with John "
)

/tmp/ipykernel_23706/1191202182.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools)


The attendees listed in **notes.txt** are:

- Priya  
- John  
- Aisha


## Multi-server agent with MultiServerMCPClient

In [23]:
from langchain_mcp_adapters.client import MultiServerMCPClient

multi_client = MultiServerMCPClient({
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(SANDBOX_DIR)],
        "transport": "stdio",
    },
    "fetch": {
        "command": "python3",
        "args": ["-m", "mcp_server_fetch"],
        "transport": "stdio",
    },
})

all_tools = await multi_client.get_tools()
print("Combined tools:", [t.name for t in all_tools])

Combined tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories', 'fetch']


In [26]:
multi_agent = create_react_agent(model, all_tools)

result = await multi_agent.ainvoke({
    "messages": (
        "Fetch the content at https://example.com, write a two-sentence "
        "summary of it, and save that summary to a file called web_summary.txt "
        "in the allowed directory."
    )
})
print(result["messages"][-1].content)

/tmp/ipykernel_23706/1898700417.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  multi_agent = create_react_agent(model, all_tools)
Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "/home/ronald/Desktop/python-pratice/ai-training/week-6/.venv/lib/python3.12/site-packages/mcp/client/stdio/__init__.py", line 155, in stdout_reader
    message = types.JSONRPCMessage.model_validate_json(line)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ronald/Desktop/python-pratice/ai-training/week-6/.venv/lib/python3.12/site-packages/pydantic/main.py", line 782, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for JSONRPCMessage


The summary has been written to **web_summary.txt** in the allowed directory.


In [27]:
# Verify the multi-server task actually happened, not just claimed to happen
summary_file = SANDBOX_DIR / "web_summary.txt"
if summary_file.exists():
    print(summary_file.read_text())
else:
    print("File not found -- check the agent's response above for what it actually did.")

The page explains that example.com is a reserved domain intended for use in documentation and examples, and should not be used for real operations. It also provides a link to IANA for more information.
